# Jewelry Pricing Assistant — Live Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bocchi277/jewelry-pricing-assistant/blob/main/Jewelry_Pricing_Assistant_Demo.ipynb)

---

This notebook demonstrates the **Jewelry Pricing Assistant** — a deterministic pricing engine for jewelry styles, with AI-generated plain-English explanations layered on top.

**Key design principles:**
- Every dollar figure is traceable to an input or a formula
- The AI (Gemini) is used **solely to describe** numbers that have already been computed
- The tool runs fully **without** an API key (deterministic fallback)
- Bad data never crashes the batch — it produces warnings instead

---

### Table of Contents
1. [Setup & Installation](#setup)
2. [Demo 1: Single Style Pricing](#demo1)
3. [Demo 2: Full Batch Processing (10 rows)](#demo2)
4. [Demo 3: Error Handling Demo](#demo3)
5. [Demo 4: AI-Powered Explanations (with Gemini)](#demo4)
6. [Unit Tests](#tests)
7. [Architecture Overview](#architecture)

<a id='setup'></a>
## 1. Setup & Installation

Clone the repository and install dependencies. This takes about 15–20 seconds.

In [ ]:
# Clone the repository
!git clone https://github.com/bocchi277/jewelry-pricing-assistant.git 2>/dev/null || echo "Repository already cloned."

# Change into the project directory
import os
os.chdir('jewelry-pricing-assistant')
print(f"Working directory: {os.getcwd()}")

# Install dependencies
!pip install -q pandas google-genai pydantic python-dotenv pytest

print("\nSetup complete!")

### Verify project structure

In [ ]:
import os

# Ensure we're in the right directory
if not os.path.exists('calculator.py'):
    os.chdir('jewelry-pricing-assistant')

print("Project Structure:")
print("=" * 50)
for item in sorted(os.listdir('.')):
    if item.startswith('.') or item == 'venv' or item == '__pycache__':
        continue
    if os.path.isdir(item):
        print(f"  [DIR] {item}/")
        for sub in sorted(os.listdir(item)):
            if not sub.startswith('.'):
                print(f"      └── {sub}")
    else:
        print(f"  [FILE] {item}")

### Preview the input data

In [ ]:
import pandas as pd

print("\nmetal_prices.csv (reference data):")
print("=" * 50)
metals_df = pd.read_csv('data/metal_prices.csv')
display(metals_df)

print("\npricing_inputs.csv (10 jewelry items to price):")
print("=" * 80)
pricing_df = pd.read_csv('data/pricing_inputs.csv')
display(pricing_df)

---

<a id='demo1'></a>
## 2. Demo 1 — Single Style Pricing

Price a single item (`B401400-14WVS`) and verify it matches the assignment's worked example **exactly**.

**Expected values from the spec:**
- Metal cost: `4.8g × $42/g = $201.60`
- Diamond cost: `1.2ct × $475/ct = $570.00`
- Total cost: `$906.60`
- Wholesale: `$906.60 × (1 + 220/100) = $2,901.12`
- Retail: `$2,901.12 × 2 = $5,802.24`

In [ ]:
import json

# Run the CLI for a single style
!python main.py --no-ai --style B401400-14WVS --output outputs/demo_single.json

# Display the result
with open('outputs/demo_single.json') as f:
    result = json.load(f)

print("\n" + "=" * 60)
print("RESULT for B401400-14WVS")
print("=" * 60)
print(json.dumps(result, indent=2))

# Verify against the worked example
r = result[0]
print("\n" + "=" * 60)
print("VERIFICATION against assignment spec:")
print("=" * 60)
checks = [
    ("Metal cost",      r['metal_cost'],      201.6),
    ("Diamond cost",    r['diamond_cost'],     570.0),
    ("Color stone cost",r['color_stone_cost'], 0.0),
    ("Labor cost",      r['labor_cost'],       95.0),
    ("Setting cost",    r['setting_cost'],     40.0),
    ("Total cost",      r['total_cost'],       906.6),
    ("Wholesale price", r['wholesale_price'],  2901.12),
    ("Retail price",    r['retail_price'],     5802.24),
]
all_pass = True
for label, actual, expected in checks:
    status = "[PASS]" if actual == expected else "[FAIL]"
    if actual != expected:
        all_pass = False
    print(f"  {status} {label:20s}  actual={actual:<12}  expected={expected}")

print(f"\n{'SUCCESS: All values match the spec exactly!' if all_pass else 'WARNING: Some values differ!'}")
print(f"\nExplanation: {r['pricing_explanation']}")
print(f"Warnings: {r['validation_warnings']}")

---

<a id='demo2'></a>
## 3. Demo 2 — Full Batch Processing (10 rows)

Process **all 10 items** from `pricing_inputs.csv` in one run. Demonstrates:
- Natural diamond vs. lab-grown detection
- Diamond + color stone combination pieces
- Platinum pricing
- Correct cost-driver identification for each item

In [ ]:
import json

# Process all 10 rows
!python main.py --no-ai --output outputs/demo_all.json

with open('outputs/demo_all.json') as f:
    results = json.load(f)

print(f"\nProcessed {len(results)} items successfully!\n")
print(f"{'Style':<24} {'Metal':>6} {'Diamond':>9} {'ColorSt':>9} {'Total':>10} {'Wholesale':>12} {'Retail':>12}  Driver")
print("─" * 120)

for r in results:
    # Extract the driver from the explanation
    expl = r['pricing_explanation']
    driver = expl.split('is ')[1].split('.')[0] if 'is ' in expl else '?'
    print(
        f"{r['style_number']:<24} "
        f"${r['metal_cost']:>8,.2f} "
        f"${r['diamond_cost']:>8,.2f} "
        f"${r['color_stone_cost']:>8,.2f} "
        f"${r['total_cost']:>9,.2f} "
        f"${r['wholesale_price']:>11,.2f} "
        f"${r['retail_price']:>11,.2f}  "
        f"{driver}"
    )

### Lab-Grown vs Natural Diamond Detection

In [ ]:
import json

with open('outputs/demo_all.json') as f:
    results = json.load(f)

print("Diamond Type Detection:")
print("=" * 60)
for r in results:
    if r['diamond_cost'] > 0:
        is_lab = 'lab-grown' in r['pricing_explanation'].lower()
        is_natural = 'natural' in r['pricing_explanation'].lower()
        label = 'Lab-grown' if is_lab else 'Natural' if is_natural else 'Unknown'
        prefix = r['style_number'][:2]
        print(f"  {r['style_number']:<24} → {label}  (prefix: {prefix})")

---

<a id='demo3'></a>
## 4. Demo 3 — Error Handling

Process `error_handling_demo.csv` which deliberately contains **every kind of bad data** the tool can handle:
- Missing markup percentage
- Unknown metal code
- Negative weights
- Non-numeric values
- Missing style numbers
- Duplicate style numbers
- Color stone carat without type

**None of these crash the batch.** Each produces a clear validation warning.

In [ ]:
import json

print("Error-handling demo input:")
print("=" * 80)
demo_df = pd.read_csv('data/error_handling_demo.csv')
display(demo_df)

# Process it
!python main.py --no-ai --pricing data/error_handling_demo.csv --output outputs/demo_errors.json

with open('outputs/demo_errors.json') as f:
    error_results = json.load(f)

print(f"\nSUCCESS: All {len(error_results)} rows processed without crashing!\n")
print("=" * 80)

for r in error_results:
    print(f"\nStyle: {r['style_number']}")
    print(f"   Retail: ${r['retail_price']:,.2f}")
    if r['validation_warnings']:
        print(f"   Warnings ({len(r['validation_warnings'])}):\n")
        for w in r['validation_warnings']:
            print(f"      • {w}")
    else:
        print("   No warnings")

---

<a id='demo4'></a>
## 5. Demo 4 — AI-Powered Explanations (with Gemini)

This section shows the difference between **deterministic fallback** text and **AI-generated** explanations.

To use AI explanations, paste your free Gemini API key below. Get one at: https://aistudio.google.com/app/apikey

> **Without a key**, the tool still works perfectly — you just get template-based explanations instead of AI prose.

In [ ]:
import os
import json

# ═══════════════════════════════════════════════════════════════
#  PASTE YOUR GEMINI API KEY BELOW (or leave blank to skip AI)
# ═══════════════════════════════════════════════════════════════
GEMINI_API_KEY = ""  # <-- paste your key here
# ═══════════════════════════════════════════════════════════════

if GEMINI_API_KEY:
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    print("API key set! Running WITH AI...\n")

    # Run with AI
    !python main.py --style B401400-14WVS --output outputs/demo_ai.json

    with open('outputs/demo_ai.json') as f:
        ai_result = json.load(f)

    # Also run without AI for comparison
    !python main.py --no-ai --style B401400-14WVS --output outputs/demo_noai.json

    with open('outputs/demo_noai.json') as f:
        noai_result = json.load(f)

    print("\n" + "=" * 70)
    print("COMPARISON: AI vs Deterministic (same item B401400-14WVS)")
    print("=" * 70)

    print(f"\nAI-generated explanation:")
    print(f"   {ai_result[0]['pricing_explanation']}")

    print(f"\nDeterministic fallback:")
    print(f"   {noai_result[0]['pricing_explanation']}")

    print(f"\nPrices are IDENTICAL (AI never changes numbers):")
    for key in ['metal_cost', 'diamond_cost', 'total_cost', 'wholesale_price', 'retail_price']:
        ai_val = ai_result[0][key]
        det_val = noai_result[0][key]
        match = "[MATCH]" if ai_val == det_val else "[MISMATCH]"
        print(f"   {match} {key:20s}  AI=${ai_val:<12}  Deterministic=${det_val}")

else:
    print("SKIPPED: No API key provided — skipping AI demo.")
    print("   The tool works perfectly without AI. All pricing is deterministic.")
    print("   AI only rephrases the explanation text — it never changes any numbers.")
    print("\n   To try AI: paste a free Gemini API key in the cell above and re-run.")
    print("   Get a free key at: https://aistudio.google.com/app/apikey")

---

<a id='tests'></a>
## 6. Unit Tests

Run the full test suite. All tests run **offline** (no API key needed) and cover:
- Calculator formulas (`test_calculator.py`)
- Input validation & coercion (`test_validator.py`)
- End-to-end integration against real CSVs (`test_integration.py`)
- Dollar-figure sanity check (Task 6)
- Auth-error detection (Task 5)
- Whitespace-aware duplicate detection (Task 3)
- Empty CSV guard (Task 7)

In [ ]:
!python -m pytest tests/ -v --tb=short 2>&1

---

<a id='architecture'></a>
## 7. Architecture Overview

### Module Responsibilities

| Module | Responsibility | Uses AI? |
|--------|---------------|----------|
| `calculator.py` | All 6 pricing formulas, metal lookup, cost-driver detection | No |
| `validator.py` | Input coercion, warning detection, template rendering | No |
| `ai_explainer.py` | Gemini API calls for natural-language explanations | Yes (Optional) |
| `main.py` | CLI, CSV loading, orchestration, JSON output | No |

### Pricing Formulas
```
metal_cost        = gold_weight_grams × price_per_gram
diamond_cost      = diamond_carat × diamond_cost_per_carat  (0 if carat is 0)
color_stone_cost  = color_stone_carat × cost_per_carat      (0 if type is blank)
total_cost        = metal + diamond + color_stone + labor + setting
wholesale_price   = total_cost × (1 + markup_percent / 100)
retail_price      = wholesale_price × 2
```

### AI Safety Guardrails
1. **Structured schema** — Pydantic model forces exact JSON shape
2. **Dollar-figure sanity check** — Any `$X.XX` in AI output is verified against computed values
3. **Warning count check** — AI must return exactly N warnings (matching detected count)
4. **Deterministic fallback** — If AI fails for any reason, template text is used
5. **Auth fast-fail** — First auth error disables AI for the rest of the run

### Interactive: Try your own inputs

In [ ]:
# You can also use the modules directly in Python:
import calculator
import validator
import json

# Build the metal lookup from the CSV data
metal_rows = [
    {"metal_group": "14K", "metal_codes": "14W,14Y,14R", "price_per_gram": 42},
    {"metal_group": "18K", "metal_codes": "18W,18Y,18R", "price_per_gram": 57},
    {"metal_group": "PT",  "metal_codes": "PT",          "price_per_gram": 38},
]
metal_lookup = calculator.build_metal_lookup(metal_rows)

# Price a custom item
metal_group, price_per_gram, _ = calculator.resolve_metal("14W", metal_lookup)

costs = calculator.calculate_costs(
    gold_weight_grams=5.0,
    price_per_gram=price_per_gram,
    diamond_carat=1.0,
    diamond_cost_per_carat=500,
    color_stone_type=None,
    color_stone_carat=0,
    color_stone_cost_per_carat=0,
    labor_cost=100,
    setting_cost=50,
    markup_percent=200,
)

driver = calculator.biggest_driver(costs)
label = calculator.driver_label(driver, metal_group, None, "CUSTOM-001", "")

print("Custom Pricing Calculation:")
print("=" * 50)
print(f"  Metal ({metal_group}):    ${costs['metal_cost']:.2f}")
print(f"  Diamond:          ${costs['diamond_cost']:.2f}")
print(f"  Color Stone:      ${costs['color_stone_cost']:.2f}")
print(f"  Labor:            ${costs['labor_cost']:.2f}")
print(f"  Setting:          ${costs['setting_cost']:.2f}")
print(f"  ──────────────────────────")
print(f"  Total Cost:       ${costs['total_cost']:.2f}")
print(f"  Wholesale (200%): ${costs['wholesale_price']:.2f}")
print(f"  Retail (×2):      ${costs['retail_price']:.2f}")
print(f"\n  Biggest cost driver: {label}")

---

## Summary

This demo has shown:

| Feature | Status |
|---------|--------|
| Deterministic pricing formulas | Verified against spec |
| All 10 input rows processed | No crashes |
| Lab-grown vs natural diamond detection | Correct |
| Color stone + diamond combination pieces | Both costs computed |
| Robust error handling (7 bad-data scenarios) | Clear warnings, no crashes |
| AI explanations (optional, Gemini) | Never changes numbers |
| Dollar-figure sanity check | Tested |
| Full test suite | All passing |

---

*Generated for the Jewelry Pricing Assistant project by bocchi277*